In [ ]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


warnings.filterwarnings('ignore')

In [38]:
df = pd.read_csv('../data/desafio_nps_fase_1.csv')
df.head()

,customer_id,customer_age,customer_region,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
0,1,63,Nordeste,14,50001,139.73,4,39.35,4,2,2,55.53,3,0,4,6.9,0,3,6.5
1,2,20,Sul,1,50002,458.95,2,9.51,10,6,4,28.23,3,0,10,2.4,0,3,0.0
2,3,46,Nordeste,111,50003,507.06,5,42.82,6,6,1,40.99,1,4,5,4.8,0,7,1.5
3,4,52,Centro-Oeste,117,50004,302.19,2,19.58,9,5,2,35.24,3,1,11,5.9,0,4,0.3
4,5,56,Norte,50,50005,253.06,1,29.37,11,13,1,39.32,1,1,0,6.1,0,3,7.9


In [48]:
# One Hot Encode a coluna 'customer_region'
one_hot_encoded_regions = pd.get_dummies(df['customer_region'], prefix='region', drop_first=True)

# Features de treinamento
X = df.drop(columns=[
    'customer_id',
    'order_id',
    'customer_region',
    'delivery_delay_days',
    'nps_score'])

# Join das features com as colunas one-hot encoded
X = X.join(one_hot_encoded_regions)
y = df['delivery_delay_days'] # Labels resultado

# Divisão dos dados em treino e teste para ambos os conjuntos de labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Linear Regression

In [49]:
linear_regression = LinearRegression()
linear_regression.fit(X_train, y_train)
linear_regression_predictions = linear_regression.predict(X_test)

# Avaliar a performance do modelo
mae_lr = mean_absolute_error(y_test, linear_regression_predictions)
rmse_lr = np.sqrt(mean_squared_error(y_test, linear_regression_predictions))
r2_lr = r2_score(y_test, linear_regression_predictions)

print("=== Resultados da Avaliação ===")
print(f"MAE (Erro Médio Absoluto): {mae_lr:.2f} dias")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse_lr:.2f} dias")
print(f"R² (Score de Variância): {r2_lr:.2f}\n")

=== Resultados da Avaliação ===
MAE (Erro Médio Absoluto): 0.59 dias
RMSE (Raiz do Erro Quadrático Médio): 0.82 dias
R² (Score de Variância): 0.68



## Decision Tree Regressor

In [50]:
decition_tree = DecisionTreeRegressor(random_state=42)
decition_tree.fit(X_train, y_train)
decition_tree_predictions = decition_tree.predict(X_test)

# Avaliar a performance do modelo
mae_dt = mean_absolute_error(y_test, decition_tree_predictions)
rmse_dt = np.sqrt(mean_squared_error(y_test, decition_tree_predictions))
r2_dt = r2_score(y_test, decition_tree_predictions)

print("=== Resultados da Avaliação ===")
print(f"MAE (Erro Médio Absoluto): {mae_dt:.2f} dias")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse_dt:.2f} dias")
print(f"R² (Score de Variância): {r2_dt:.2f}\n")

# Extrair as features mais importantes para o atraso
importancias = pd.DataFrame({
    'Variável': X_train.columns, 
    'Importância': decition_tree.feature_importances_
}).sort_values(by='Importância', ascending=False)

print("=== Top 5 Variáveis Mais Impactantes ===")
print(importancias.head(5))

=== Resultados da Avaliação ===
MAE (Erro Médio Absoluto): 0.80 dias
RMSE (Raiz do Erro Quadrático Médio): 1.14 dias
R² (Score de Variância): 0.38

=== Top 5 Variáveis Mais Impactantes ===
                     Variável  Importância
13        csat_internal_score     0.367748
10       resolution_time_days     0.220898
9   customer_service_contacts     0.152537
7               freight_value     0.041510
2                 order_value     0.033471


## Random Forest Regressor

In [51]:
random_forest = RandomForestRegressor(random_state=42)
random_forest.fit(X_train, y_train)
random_forest_predictions = random_forest.predict(X_test)

# Avaliar a performance do modelo
mae_rf = mean_absolute_error(y_test, random_forest_predictions)
rmse_rf = np.sqrt(mean_squared_error(y_test, random_forest_predictions))
r2_rf = r2_score(y_test, random_forest_predictions)

print("=== Resultados da Avaliação ===")
print(f"MAE (Erro Médio Absoluto): {mae_rf:.2f} dias")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse_rf:.2f} dias")
print(f"R² (Score de Variância): {r2_rf:.2f}\n")

# Extrair as features mais importantes para o atraso
importancias = pd.DataFrame({
    'Variável': X_train.columns, 
    'Importância': random_forest.feature_importances_
}).sort_values(by='Importância', ascending=False)

print("=== Top 5 Variáveis Mais Impactantes ===")
print(importancias.head(5))

=== Resultados da Avaliação ===
MAE (Erro Médio Absoluto): 0.58 dias
RMSE (Raiz do Erro Quadrático Médio): 0.78 dias
R² (Score de Variância): 0.71

=== Top 5 Variáveis Mais Impactantes ===
                     Variável  Importância
13        csat_internal_score     0.356075
10       resolution_time_days     0.224141
9   customer_service_contacts     0.142383
1      customer_tenure_months     0.038350
2                 order_value     0.032621


Baseado nas métricas de avaliação de cada modelo, percebemos que o Random Forest Regressor é o melhor modelo que se adapta ao problema proposto.